In [6]:
# =====================================================
# Cell 1 : 라이브러리 임포트
# =====================================================
import re
import pandas as pd


# =====================================================
# Cell 2 : 경로 설정
# =====================================================
INPUT_CSV  = "../../00_data/00_raw/coos_성분정보.csv"
OUTPUT_JSON = "../../00_data/02_processed/coos_ewg_cleaned.json"
OUTPUT_CSV  = "../../00_data/02_processed/coos_ewg_cleaned.csv"


# =====================================================
# Cell 3 : CSV 로드 및 컬럼 확인
# =====================================================
df_raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
print(f"총 행 수: {len(df_raw):,}")
print(f"컬럼 목록: {df_raw.columns.tolist()}")
print()
df_raw.head(3)


# =====================================================
# Cell 4 : 컬럼명 지정
# ↓ Cell 3 출력 보고 실제 컬럼명으로 수정
# =====================================================
ING_COL   = "성분명"      # ← 성분명 컬럼명으로 수정
SCORE_COL = "스코어"      # ← 스코어 컬럼명으로 수정


# =====================================================
# Cell 5 : 스코어 파싱 함수 (수정 - None → 0)
# =====================================================
def parse_score(raw):
    if raw is None:
        return 0

    raw_str = str(raw).strip()

    if raw_str in ("", "nan", "None", "N/A", "-"):
        return 0

    cleaned = re.sub(r"[^\d\-–]", " ", raw_str).strip()
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # 범위 패턴: "3-9" → 9 (끝값)
    range_match = re.match(r"^(\d+)\s*[-–]\s*(\d+)$", cleaned)
    if range_match:
        return int(range_match.group(2))

    # 단일 숫자
    single_match = re.match(r"^(\d+)$", cleaned)
    if single_match:
        return int(single_match.group(1))

    # 숫자 여러 개 중 마지막
    numbers = re.findall(r"\d+", cleaned)
    if numbers:
        return int(numbers[-1])

    return 0  # 파싱 불가도 0

# =====================================================
# Cell 6 : 파싱 + 정제
# =====================================================
df_raw["score_parsed"] = df_raw[SCORE_COL].apply(parse_score)

# 성분명 비어있는 행 제거
df_raw = df_raw[
    df_raw[ING_COL].notna() &
    (df_raw[ING_COL].astype(str).str.strip() != "")
].copy()

df_raw["ingredient_key"] = df_raw[ING_COL].astype(str).str.strip().str.lower()

print(f"파싱 완료: {len(df_raw):,}행")
print(f"유효값: {df_raw['score_parsed'].notna().sum():,}개")
print(f"NaN:    {df_raw['score_parsed'].isna().sum():,}개")
print("\n점수 분포:")
print(df_raw["score_parsed"].value_counts(dropna=False).sort_index())


# =====================================================
# Cell 7 : 중복 성분명 병합 (수정 - None → 0)
# =====================================================
merged_rows = []

for key, group in df_raw.groupby("ingredient_key"):
    # 0 아닌 유효값 우선, 없으면 0
    valid_scores = [s for s in group["score_parsed"].tolist() if s != 0]
    score = valid_scores[0] if valid_scores else 0

    representative = group[ING_COL].astype(str).str.strip().value_counts().idxmax()

    merged_rows.append({
        "ingredient": representative,
        "coos_score": int(score),
    })

df = pd.DataFrame(merged_rows).sort_values("ingredient").reset_index(drop=True)

print(f"병합 전: {len(df_raw):,}행")
print(f"병합 후: {len(df):,}행 (고유 성분 수)")
print(f"유효값(0 제외): {(df['coos_score'] != 0).sum():,}개")
print(f"0(null/공백):   {(df['coos_score'] == 0).sum():,}개")
print("\n점수 분포:")
print(df["coos_score"].value_counts(dropna=False).sort_index())
df.head(10)

# =====================================================
# Cell 8 : 저장
# =====================================================
import json

# CSV
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"CSV 저장: {OUTPUT_CSV}")

# JSON (NaN 제거, 유효값만)
out = (
    df.dropna(subset=["coos_score"])
    .assign(coos_score=lambda x: x["coos_score"].astype(int))
    .to_dict(orient="records")
)
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print(f"JSON 저장: {OUTPUT_JSON}")
print(f"최종 성분 수 (NaN 제외): {len(out):,}")

총 행 수: 18,179
컬럼 목록: ['성분명', 'URL', 'INCI', '한글명', '기능', '종류', '이명', '구명칭', 'CAS No.', 'EC No.', '구조식', '유럽 CosIng Ref No.', '국가', '🇰🇷국내', '🇨🇳중국', '🇹🇼대만', '🇯🇵일본', '🇩🇪유럽', '🇻🇳아세안', 'AI설명', '스코어', '데이터 등급', '링크']

파싱 완료: 18,179행
유효값: 18,179개
NaN:    0개

점수 분포:
score_parsed
0     10902
1      5121
2      1150
3       453
4       241
5       117
6        92
7        39
8        13
9        16
10       35
Name: count, dtype: int64
병합 전: 18,179행
병합 후: 18,178행 (고유 성분 수)
유효값(0 제외): 7,277개
0(null/공백):   10,901개

점수 분포:
coos_score
0     10901
1      5121
2      1150
3       453
4       241
5       117
6        92
7        39
8        13
9        16
10       35
Name: count, dtype: int64
CSV 저장: ../../00_data/02_processed/coos_ewg_cleaned.csv
JSON 저장: ../../00_data/02_processed/coos_ewg_cleaned.json
최종 성분 수 (NaN 제외): 18,178
